# Act I — Primer: text → graph

> *"Looks easy, right? Here's where everyone falls in."*

Building a graph by **extraction**: pull structured data — entities + relationships — out of
unstructured text. This is the fast, fun peak of the curve.

**[HOW · recipe, live]** We take raw recipe text and turn it into typed records, then build the graph (v1):

- `Recipe -[CONTAINS {qty, unit}]-> Ingredient`
- `Recipe -[HAS_STEP]-> Step`
- `Step -[TECHNIQUE]-> Technique`

Recipe model **v1** — already more than a contains-at-quantity star, but still naive
(no resolution, raw units, duplicate ingredients). That's the valley waiting in Act II.

Everything here runs **offline**: extraction replays from a committed cache in `data/cache/`,
so there's no API key and no network call on stage.

## v0 — extract with NO schema (the naive bottom)

_New Act I opening: the bottom of the staircase. (The schema'd v1 below is now Act II rung 1.)_

The naive move everyone makes: ask an LLM for entities + relationships, **no schema, no ontology,
no resolution** — just free-form `(subject, predicate, object)` triples. You get a graph… but it's a mess.


In [ ]:
from IPython.display import Image
from graphtools.data import load_hero_texts
from graphtools.extract import extract_freeform
from graphtools.graph import build_naive_graph, nodes_of
from graphtools.viz import render_graph

title, text = next(t for t in load_hero_texts() if t[0] == "Pancakes")
triples = extract_freeform(text)  # v0, offline replay
print(f"{title}: {len(triples)} free-form triples (no schema). e.g.")
for t in triples[:5]:
    print("   ", t["subject"], "-[", t["predicate"], "]->", t["object"])

g0 = build_naive_graph(triples)
print("naive graph:", g0, "| untyped 'entity' nodes:", len(nodes_of(g0, "entity")))
render_graph(g0, path="act1_v0_naive.png")
Image(filename="act1_v0_naive.png")


Look at the mess: quantities mashed into entity names (`100 g Flour`), a different relation label
almost every time, **no node types, nothing resolved**. That's the bottom — *"look, a knowledge
graph!"* that you can't actually trust. Act II climbs out: **add shape → ontology → matching.**

In [ ]:
from graphtools.data import load_hero_texts
from graphtools.extract import extract_recipe
from graphtools.graph import build_graph, nodes_of
from graphtools.viz import render_graph

# The hero set: ~14 real recipes (TheMealDB), committed as JSON — our deterministic corpus.
hero_texts = load_hero_texts()  # list of (title, raw_text)
print(f"{len(hero_texts)} hero recipes loaded\n")

# Here's one raw recipe — unstructured text, the kind the world actually hands you.
title, text = hero_texts[0]
print(f"=== {title} ===\n")
print(text)

## Text → structured records

`extract_recipe` runs a schema-first LLM extraction (replayed from cache here).
Out comes a typed `Recipe` (pydantic): ingredients with quantities and units, steps with techniques.
Unstructured prose became data you can build on.

In [ ]:
# Extract every hero recipe (OFFLINE replay — the cache is committed, no API key needed).
recipes = [extract_recipe(t) for _, t in hero_texts]
print(f"extracted {len(recipes)} recipes\n")

# One resulting Recipe, to make "text → structured records" tangible:
one = recipes[0]
print(f"{one.title}: {len(one.ingredients)} ingredients, {len(one.steps)} steps\n")
one

## Records → graph

`build_graph` turns the records into a NetworkX graph: typed nodes (Recipe, Ingredient, Step,
Technique) wired by typed edges. *Structure is data* — and now we can count it, traverse it, draw it.

In [ ]:
# Build the graph over the FULL hero set.
g = build_graph(recipes)
print(f"nodes: {g.number_of_nodes()}   edges: {g.number_of_edges()}\n")

# A sample of the ingredient nodes — note the duplicates / surface-form drift (e.g. flour vs
# plain flour). That mess is exactly what Act II's resolution step cleans up.
ingredients = nodes_of(g, "ingredient")
print(f"{len(ingredients)} ingredient nodes; sample:")
ingredients[:15]

## Look, a graph

The payoff slide. We render a **single** recipe's graph (the full set is too dense to read live):
nodes coloured by kind, edges labelled by relationship. This is the cool picture everyone falls for —
and the whole point of Act I is that getting *here* is the easy part.

In [ ]:
from IPython.display import Image

# Render one recipe's graph to a PNG and show it inline (viz uses a headless Agg backend).
png_path = "act1_graph.png"
render_graph(build_graph([one]), path=png_path)
Image(filename=png_path)

---

**[MONEY · judgements]** The same move on a real corpus — extract entities from case law
into a graph. *Show, don't type:* this is a pre-built, at-scale artifact, toured in **Act III**,
not live-coded here.

> *"Looks easy, right? Here's where everyone falls in."* → **Act II: the valley.**